# TrOCR Beginner Notebook

A step-by-step starter for handwritten OCR with TrOCR.

## What you will do
1. Install tools
2. Run a tiny sample inference
3. Load CSV-based data
4. Fine-tune on your own dataset later

In [ ]:
!pip -q install transformers accelerate datasets evaluate jiwer pillow pandas scikit-learn sentencepiece

In [ ]:

import os
from pathlib import Path
import pandas as pd
from PIL import Image
import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel


## Project paths

In [ ]:

BASE_DIR = Path('/content/drive/MyDrive/trocr_beginner_project')
SAMPLE_DIR = BASE_DIR / 'data' / 'sample'
IMG_DIR = SAMPLE_DIR / 'images'
CSV_PATH = SAMPLE_DIR / 'sample.csv'


## Sample test files

In [ ]:

# This CSV is for pipeline testing only.
# Later you will replace it with IAM or your own handwritten form crops.
df = pd.read_csv(CSV_PATH)
df


## Load TrOCR

In [ ]:

MODEL_NAME = 'microsoft/trocr-base-handwritten'
processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)


## Run inference on a sample image

In [ ]:

img_path = IMG_DIR / df.loc[0, 'image_path']
image = Image.open(img_path).convert('RGB')
pixel_values = processor(images=image, return_tensors='pt').pixel_values.to(device)

with torch.no_grad():
    generated_ids = model.generate(pixel_values, max_length=64)

pred = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
print('Predicted text:', pred)
image


## Next step

In [ ]:

# Replace SAMPLE_DIR with your real IAM or form-field dataset.
# The training pipeline expects CSV columns: image_path,text
